# AEM4L2 E01 - Las 4 etapas de un audio pipeline

**Objetivo:** entender desde cero qué pasa entre un archivo de audio y una decisión automática.

Este notebook es el mapa mental de toda la clase. Antes de hablar de WER, Pydantic o LangChain, necesitamos ver el pipeline completo en capas.

**Python exercise relacionado:** `python_puro/AEM4_python_exercises/AEM4L2_audio_pipelines/e01_audio_transcripcion_basica.py`


## Idea principal

Un sistema de audio no es solamente "transcribir". Transcribir es una etapa. Un pipeline completo suele tener 4 etapas:

| Etapa | Pregunta que responde | Output típico |
|---|---|---|
| 1. Audio input + preprocesamiento | ¿El audio se puede escuchar bien? | WAV normalizado, limpio o segmentado |
| 2. ASR | ¿Qué texto cree el modelo que se dijo? | Transcript / hypothesis |
| 3. Post-procesamiento con LLM | ¿Qué significa o qué hay que hacer? | Resumen, intent, action items |
| 4. Evaluación + decisión | ¿Confiamos en el resultado? | WER, riesgo, gate, revisión humana |

Frase docente: **un pipeline confiable no termina cuando devuelve texto; termina cuando decide si ese texto sirve para actuar.**


In [ ]:
stages = [
    ("1. Audio", "WAV / llamada / nota de voz", "¿se escucha bien?"),
    ("2. ASR", "transcripción cruda", "¿qué dijo la persona?"),
    ("3. LLM", "resumen + acciones", "¿qué significa para el negocio?"),
    ("4. Gate", "aceptar o revisar", "¿podemos automatizar?")
]

for name, output, question in stages:
    print(f"{name:<10} -> {output:<25} -> {question}")


## Diagrama conceptual

```text
AUDIO WAV
  │
  ▼
[1] PREPROCESAMIENTO
    - formato
    - volumen
    - ruido
    - pausas / cortes
  │
  ▼
[2] ASR / WHISPER
    audio -> transcript
  │
  ▼
[3] LLM / POST-PROCESAMIENTO
    transcript -> summary, intent, action_items
  │
  ▼
[4] EVALUACIÓN Y GATE
    WER + errores críticos -> automatizar o revisión humana
```


## Etapa 1: Audio input + preprocesamiento

Antes del modelo, el audio ya puede venir con problemas.

| Problema cotidiano | Qué pasa en la práctica | Riesgo para ASR |
|---|---|---|
| Ruido de fondo | autos, teclado, gente hablando | el modelo inventa o confunde palabras |
| Volumen bajo | micrófono lejos | omite palabras importantes |
| Habla rápida | muchas palabras por segundo | junta palabras o saltea partes |
| Pausas largas | silencios inesperados | corta frases o cambia puntuación |
| Audio entrecortado | microcortes de llamada | pierde palabras completas |

En `data/` tenemos variantes para practicar: `_ruido`, `_rapido`, `_entrecortado`, `_pausas`, `_mal_estado`.


In [ ]:
from pathlib import Path

DATA_DIR = Path('../../../python_puro/AEM4_python_exercises/AEM4L2_audio_pipelines/data')
# Si ejecutás desde otra ubicación, esta celda igual sirve como ejemplo de nombres.
examples = [
    'llamada_soporte.wav',
    'llamada_soporte_ruido.wav',
    'llamada_soporte_rapido.wav',
    'llamada_soporte_entrecortado.wav',
    'llamada_soporte_pausas.wav',
    'llamada_soporte_mal_estado.wav',
]

print('Audios para comparar en clase:')
for name in examples:
    print('-', name)


## Etapa 2: ASR

**ASR** significa **Automatic Speech Recognition**.

Definición simple:

> ASR es el módulo que escucha audio y escribe texto.

En nuestros scripts, esa etapa la hace OpenAI Whisper mediante API real.

Importante: el ASR no "entiende el negocio". Solo produce una **hypothesis**, es decir, una transcripción candidata.


## Etapa 3: post-procesamiento con LLM

Después de ASR, tenemos texto. Pero un agente o un backend normalmente necesita algo más útil:

| Transcript crudo | Salida accionable |
|---|---|
| "El pedido no llegó y lo necesito hoy" | `intent = delivery_claim` |
| "Necesito una solución rápida" | `urgency = high` |
| "Pueden enviarlo o cancelar" | `action_items = [...]` |

Frase docente: **el transcript es evidencia; la estructura es operación.**


In [ ]:
transcript = "Hola, llamo porque el pedido 4521 no llegó y lo necesitaba para hoy."
structured = {
    "summary": "El cliente reclama que el pedido 4521 no llegó.",
    "intent": "delivery_claim",
    "urgency": "high",
    "action_items": ["Verificar estado del envío", "Contactar al cliente"],
}

print('TRANSCRIPT:')
print(transcript)
print()
print('ESTRUCTURA:')
for key, value in structured.items():
    print(f'{key}: {value}')


## Etapa 4: evaluación + decisión

Una mala transcripción puede producir un resumen muy convincente pero incorrecto.

Por eso medimos:

- **WER:** cuántas palabras salieron mal.
- **Errores críticos:** si se equivocó una palabra peligrosa para el dominio.
- **Gate:** regla que decide si se automatiza o se manda a revisión humana.

```text
transcripción confiable -> resumen automático
transcripción dudosa    -> revisión humana
```


In [ ]:
cases = [
    {"case": "audio claro", "wer": 0.02, "critical_error": False},
    {"case": "ruido moderado", "wer": 0.18, "critical_error": False},
    {"case": "dosis médica mal", "wer": 0.06, "critical_error": True},
    {"case": "audio cortado", "wer": 0.42, "critical_error": False},
]

for row in cases:
    decision = "revisión humana" if row["wer"] > 0.20 or row["critical_error"] else "automatizar"
    print(f"{row['case']:<20} WER={row['wer']:.0%} critical={row['critical_error']} -> {decision}")


## Mini cierre

Lo que vimos:

```text
audio -> ASR -> transcript -> LLM -> estructura -> evaluación -> decisión
```

Pregunta para el aula:

> Si el resumen final está mal, ¿dónde pudo haber nacido el problema: en el audio, en ASR, en el LLM o en el gate?
